In [ ]:
import os
import sys
from pathlib import Path

# 1. Safely locate the repository root (handles different Jupyter launch locations)
current_dir = Path().resolve()
if current_dir.name == "notebooks":
    REPO_ROOT = current_dir.parent
else:
    REPO_ROOT = current_dir

# 2. Define strict local data paths
PROJECT_DATA_DIR = REPO_ROOT / "data"
RAW_DATA_DIR = PROJECT_DATA_DIR / "raw"
PROCESSED_DATA_DIR = PROJECT_DATA_DIR / "processed"

# Ensure the directories exist
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# 3. Mount the Vendor directory (your massive scraper)
SCRAPER_DIR = REPO_ROOT / "vendor" / "PolyMarketScrapping"

if SCRAPER_DIR.exists():
    os.chdir(SCRAPER_DIR)  # The scraper expects to run from its own root
    if str(SCRAPER_DIR) not in sys.path:
        sys.path.append(str(SCRAPER_DIR))
    print(f"✅ Successfully mounted Vendor Scraper from: {SCRAPER_DIR}")
    print(f"✅ Data directory mapped to: {PROJECT_DATA_DIR}")
else:
    print(f"❌ CRITICAL ERROR: Could not find scraper at {SCRAPER_DIR}")
    print("Fix: Ensure the 'PolyMarketScrapping' folder is placed inside your 'vendor' folder.")

In [ ]:
from fetcher.config import get_config

print("--- Injecting Centralized Data Paths into Vendor Scraper ---")

try:
    # 1. Load the vendor's configuration object into memory
    scraper_config = get_config()

    # 2. Force every single output directory to route to your strict local RAW_DATA_DIR
    scraper_config.output_dirs.gamma_markets = f"{RAW_DATA_DIR}/gamma_markets"
    scraper_config.output_dirs.trade = f"{RAW_DATA_DIR}/trades"
    scraper_config.output_dirs.price = f"{RAW_DATA_DIR}/prices"
    scraper_config.output_dirs.markets = f"{RAW_DATA_DIR}/markets"
    scraper_config.output_dirs.market_tokens = f"{RAW_DATA_DIR}/market_tokens"

    # (Add any other directories the scraper uses here)

    print(f"✅ Gamma Markets successfully redirected to: {scraper_config.output_dirs.gamma_markets}")
    print(f"✅ Trades successfully redirected to: {scraper_config.output_dirs.trade}")
    print("Vendor configuration safely overridden in memory. No source files were altered.")

except Exception as e:
    print(f"❌ Configuration Override Failed: {e}")
    print("Ensure you ran Cell 1 first so the vendor package is mounted in sys.path.")

In [ ]:
# Execute the fetcher module in 'gamma' mode to acquire comprehensive market metadata.
# --mode gamma: Targets the Gamma API to retrieve questions, volume, resolution status, and token IDs.
# --fresh: DELETE IF RERUN FOR UPDATE!!! Overrides existing cursors to ensure a full pull of historical resolved markets.
# This stage populates the '{RAW_DATA_DIR}gamma_markets/' directory with raw Parquet files.
!python -m fetcher.main --mode gamma --fresh

In [ ]:
import duckdb

# Initialize DuckDB
con = duckdb.connect()

print("--- Parquet File Schema & Examples ---")
try:
    # 1. Get Schema
    schema_df = con.execute("DESCRIBE SELECT * FROM read_parquet('{RAW_DATA_DIR}/gamma_markets/**/*.parquet')").df()

    # 2. Get one sample row
    sample_df = con.execute("SELECT * FROM read_parquet('{RAW_DATA_DIR}/gamma_markets/**/*.parquet') LIMIT 1").df()

    # 3. Transpose sample row to match schema structure
    # The sample_df has 1 row and N columns. We want N rows (one per column).
    sample_transposed = sample_df.T.reset_index()
    sample_transposed.columns = ['column_name', 'example_value']

    # 4. Merge schema with example values
    result = schema_df[['column_name', 'column_type']].merge(sample_transposed, on='column_name', how='left')

    # Display
    print(result.to_string(index=False))

except Exception as e:
    print(f"Error: {e}")

In [ ]:
import duckdb
import pandas as pd
import os

# Initialize DuckDB
con = duckdb.connect()

# Define the filtering logic with strict Time Window
# 1. Binary Outcomes: ["Yes", "No"] only.
# 2. Duration: > 7 days.
# 3. Resolved: closed = true.
# 4. Time Window: endDate between April 1, 2025 and July 31, 2025.
# 5. Volume: Top 778.

query = f"""
SELECT
    conditionId,
    question,
    startDate,
    endDate,
    volumeNum,
    clobTokenIds,
    outcomes,
    date_diff('day', TRY_CAST(startDate AS TIMESTAMP), TRY_CAST(endDate AS TIMESTAMP)) as duration_days
FROM read_parquet('{RAW_DATA_DIR}/gamma_markets/**/*.parquet')
WHERE closed = true
AND (outcomes = '["Yes", "No"]' OR outcomes = '["No", "Yes"]')
AND startDate != ''
AND duration_days > 7
AND TRY_CAST(endDate AS TIMESTAMP) >= '2025-04-01'
AND TRY_CAST(endDate AS TIMESTAMP) <= '2025-07-31'
ORDER BY volumeNum DESC
LIMIT 778
"""

try:
    df_targets = con.execute(query).df()

    # Fallback: If strict window yields too few results, widen to "All 2025"
    if len(df_targets) < 100:
        print(f"Strict window yielded only {len(df_targets)} markets. Widening to full year 2025...")
        query_wide = query.replace("AND TRY_CAST(endDate AS TIMESTAMP) >= '2025-04-01'", "AND TRY_CAST(endDate AS TIMESTAMP) >= '2025-01-01'")
        query_wide = query_wide.replace("AND TRY_CAST(endDate AS TIMESTAMP) <= '2025-07-31'", "AND TRY_CAST(endDate AS TIMESTAMP) <= '2025-12-31'")
        df_targets = con.execute(query_wide).df()

    # Save to centralized data folder
    output_path = PROCESSED_DATA_DIR / "target_markets.csv"
    df_targets.to_csv(output_path, index=False)

    print(f"Filtering complete.")
    print(f"Total markets matching criteria: {len(df_targets)}")
    print(f"Target list exported to: {output_path}")

    if not df_targets.empty:
        display(df_targets)

except Exception as e:
    print(f"Query failed. Error: {e}")

In [ ]:
import json
import pandas as pd
import os
from datetime import datetime

print("Preparing VIP Target List...")
df = pd.read_csv(PROCESSED_DATA_DIR / "target_markets.csv")

vip_data = {
    "trades": [],
    "prices": []
}

for _, row in df.iterrows():
    vip_data["trades"].append(row['conditionId'])
    try:
        start_ts = int(pd.Timestamp(row['startDate']).timestamp())
        end_ts = int(pd.Timestamp(row['endDate']).timestamp())
        tokens = json.loads(row['clobTokenIds'])
        for token in tokens:
            vip_data["prices"].append([token, start_ts, end_ts, True])
    except Exception as e:
        print(f"Skipping row: {e}")

with open("vip_targets.json", "w") as f:
    json.dump(vip_data, f)

runner_script = """
import json
import sys
import os
from queue import Queue
from threading import Thread

sys.path.append(os.getcwd())

from fetcher.coordination import FetcherCoordinator
from fetcher.config import get_config

def main():
    print(">>> Starting Surgical Data Acquisition (Fixed Initialization)...")

    with open("vip_targets.json", "r") as f:
        targets = json.load(f)

    config = get_config()
    config.output_dirs.trade = f"{RAW_DATA_DIR}/trades"
    config.output_dirs.price = f"{RAW_DATA_DIR}/prices"
    config.workers.trade = 2
    config.workers.price = 2

    coord = FetcherCoordinator(config=config)

    # FIX: Initialize both Queues AND Fetchers
    coord._create_queues(use_swappable=True)
    coord._create_fetchers()  # <--- This was missing!

    print(f"Injecting {len(targets['trades'])} markets into Trade Queue...")
    for market_id in targets['trades']:
        coord._trade_market_queue.put(market_id)

    print(f"Injecting {len(targets['prices'])} tokens into Price Queue...")
    for token_data in targets['prices']:
        coord._price_token_queue.put(tuple(token_data))

    for _ in range(config.workers.trade):
        coord._trade_market_queue.put(None)
    for _ in range(config.workers.price):
        coord._price_token_queue.put(None)

    print(">>> Launching Workers...")

    trade_threads = []
    for i in range(config.workers.trade):
        t = Thread(target=coord._trade_fetcher._worker, args=(i, coord._trade_output_queue))
        t.start()
        trade_threads.append(t)

    price_threads = []
    for i in range(config.workers.price):
        t = Thread(target=coord._price_fetcher._worker, args=(i, coord._price_output_queue, None, None))
        t.start()
        price_threads.append(t)

    print(">>> Workers Running...")

    for t in trade_threads: t.join()
    for t in price_threads: t.join()

    coord._stop_persisters()
    print(">>> Acquisition Complete.")

if __name__ == "__main__":
    main()
"""

with open("custom_runner.py", "w") as f:
    f.write(runner_script)

!python custom_runner.py

In [ ]:
# Install the heavy-lifting library for embeddings
!pip install -q sentence-transformers

import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Define Paths (Using the flat structure)
INPUT_FILE = os.path.join(PROCESSED_DATA_DIR, "target_markets.csv")
OUTPUT_FILE = os.path.join(PROCESSED_DATA_DIR, "clustered_markets.csv")


In [ ]:
def load_and_clean_data(filepath):
    """
    Loads market metadata and preprocesses questions for embedding.
    Removes common stop words to focus on semantic keywords.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find {filepath}")

    df = pd.read_csv(filepath)

    # Basic cleaning: Lowercase, strip whitespace
    # We remove "Will" and "?" because they appear in almost every prediction market
    # and dilute the semantic value of the vector.
    df['clean_question'] = df['question'].astype(str).str.lower().str.strip()
    df['clean_question'] = df['clean_question'].str.replace(r'^will ', '', regex=True)
    df['clean_question'] = df['clean_question'].str.replace(r'\?$', '', regex=True)

    print(f"Loaded {len(df)} markets.")
    return df

# Execute Tool A
df_markets = load_and_clean_data(INPUT_FILE)
display(df_markets[['question', 'clean_question']])

In [ ]:
def generate_embeddings(text_list):
    """
    Converts a list of text strings into a 384-dimensional vector matrix.
    Uses the 'all-MiniLM-L6-v2' model which is optimized for semantic similarity.
    """
    print("Loading SentenceTransformer model...")
    # This will download the model (~80MB) on the first run
    model = SentenceTransformer('all-MiniLM-L6-v2')

    print(f"Encoding {len(text_list)} questions...")
    embeddings = model.encode(text_list, show_progress_bar=True)

    print(f"Embedding Shape: {embeddings.shape}")
    return embeddings

# Execute Tool B
market_embeddings = generate_embeddings(df_markets['clean_question'].tolist())

In [ ]:
def cluster_markets(embeddings, n_clusters):
    """
    Groups markets into 'n_clusters' based on vector similarity.
    Uses a fixed random_state for reproducibility.
    """
    print(f"Clustering into {n_clusters} topics...")

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)

    return labels

# Execute Tool C
# Logic: K = N / 10
K = len(df_markets) // 10
print(f"Target K: {K}")

df_markets['cluster_id'] = cluster_markets(market_embeddings, K)

print("\n--- Sample Cluster 0 ---")
print(df_markets[df_markets['cluster_id'] == 0]['question'].to_string(index=False))

In [ ]:
!pip install -U -q google-generativeai

import google.generativeai as genai
from google.colab import userdata

# Configure API
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)

    # 3. Retrieve all models supporting content generation
    available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]

    # 4. Define model preference hierarchy (Flash priority)
    # We look for the highest version of Flash available in the current API environment
    preferences = ["gemini-2.5-flash-lite", "gamma-3", "gamma"]

    target_model = ""
    for pref in preferences:
        for name in available_models:
            if pref in name.lower():
                target_model = name
                break
        if target_model:
            break

    # Final fallback to any available model if no Flash models are found
    if not target_model and available_models:
        target_model = available_models[0]

    if target_model:
        model = genai.GenerativeModel(target_model)
        print(f"Successfully connected to model: {target_model}")
    else:
        print("CRITICAL ERROR: No suitable Gemini models found in your API project.")

except Exception as e:
    print(f"Configuration Error: {e}")

In [ ]:
import time
import re

def generate_cluster_labels_batched(df, batch_size=10):
    cluster_labels = {}
    unique_clusters = sorted(df['cluster_id'].unique())
    taxonomy = "politics, geopolitics, elections, economy, finance, earnings, crypto, tech, sports, culture, other"

    print(f"Labeling {len(unique_clusters)} clusters in batches of {batch_size} using {target_model}...")

    # Process in batches
    for i in range(0, len(unique_clusters), batch_size):
        batch_ids = unique_clusters[i:i + batch_size]

        # Build a massive prompt for multiple clusters
        prompt_content = ""
        for cid in batch_ids:
            subset = df[df['cluster_id'] == cid]
            samples = subset['question'].sample(min(len(subset), 5)).tolist()
            questions_block = "\n".join([f"  - {q}" for q in samples])
            prompt_content += f"CLUSTER_ID {cid}:\n{questions_block}\n\n"

        prompt = f"""
        You are a data analyst. I have several clusters of prediction market questions.
        Assign EACH cluster to exactly ONE of these categories: [{taxonomy}]

        {prompt_content}

        Return your answer as a simple list following this format:
        ID: category
        ID: category
        """

        try:
            response = model.generate_content(prompt)
            response_text = response.text.lower()

            # Parse the response using Regex to find "ID: category"
            # Matches patterns like "0: crypto" or "ID 0: crypto"
            matches = re.findall(r"(\d+):\s*(\w+)", response_text)

            for cid_str, label in matches:
                cid = int(cid_str)
                if label.strip() in taxonomy:
                    cluster_labels[cid] = label.strip()
                else:
                    cluster_labels[cid] = "other"
                print(f"Parsed Cluster {cid:02d}: {cluster_labels[cid]}")

            # Small safety sleep for RPM
            time.sleep(2)

        except Exception as e:
            print(f"Error on batch starting at {batch_ids[0]}: {e}")
            # Fallback: mark batch as other so we can continue
            for cid in batch_ids:
                if cid not in cluster_labels:
                    cluster_labels[cid] = "other"
            time.sleep(8)

    return cluster_labels

# Execute the batched version
label_mapping = generate_cluster_labels_batched(df_markets)

# Map results to dataframe
df_markets['cluster_label'] = df_markets['cluster_id'].map(label_mapping)

# Final check
missing = df_markets['cluster_label'].isna().sum()
print(f"\nFinished. Missing labels: {missing}")

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_clusters(embeddings, df):
    """
    Reduces dimensions to 2D and plots the clusters colored by LLM labels.
    """
    print("Running t-SNE (this may take a moment)...")
    tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
    vis_dims = tsne.fit_transform(embeddings)

    df['tsne_x'] = vis_dims[:, 0]
    df['tsne_y'] = vis_dims[:, 1]

    plt.figure(figsize=(12, 8))
    sns.scatterplot(
        data=df,
        x='tsne_x',
        y='tsne_y',
        hue='cluster_label',
        palette='viridis',
        alpha=0.7
    )
    plt.title("Semantic Clusters of Polymarket Questions")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()

# Execute Tool E
visualize_clusters(market_embeddings, df_markets)

# Final Step: Save the results
df_markets.to_csv(OUTPUT_FILE, index=False)
print(f"\nPhase 2 Complete. Clustered data saved to: {OUTPUT_FILE}")

In [ ]:
from sklearn.manifold import TSNE
import plotly.express as px
import pandas as pd

def visualize_clusters_3d(embeddings, df):
    """
    Reduces dimensions to 3D and creates an interactive plot.
    """
    print("Running t-SNE in 3D (this may take a moment)...")

    # 1. Run t-SNE with 3 components
    tsne = TSNE(n_components=3, random_state=42, init='pca', learning_rate='auto')
    vis_dims = tsne.fit_transform(embeddings)

    # 2. Add coordinates to DataFrame
    df['x'] = vis_dims[:, 0]
    df['y'] = vis_dims[:, 1]
    df['z'] = vis_dims[:, 2]

    # 3. Create Interactive Plot
    print("Generating 3D Plot...")
    fig = px.scatter_3d(
        df,
        x='x', y='y', z='z',
        color='cluster_label',
        hover_data=['question', 'cluster_id'], # Show question on hover
        title="3D Semantic Clusters of Polymarket Questions",
        opacity=0.7,
        size_max=5
    )

    # Tweak the layout for better visibility
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
    fig.show()

# Execute Tool E (3D Version)
visualize_clusters_3d(market_embeddings, df_markets)

# Final Step: Save the results (including 3D coords for future use)
df_markets.to_csv(OUTPUT_FILE, index=False)
print(f"\nPhase 2 Complete. Clustered data saved to: {OUTPUT_FILE}")

In [ ]:
import pandas as pd
import duckdb
import os
import json
import time
import google.generativeai as genai
from google.colab import userdata

# 1. Configure API (Using Gemma 3 for high quota)
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel("models/gemma-3-27b-it") # High quota model
    print("AI Agent Ready: Gemma 3 27B")
except Exception as e:
    print(f"API Error: {e}")

# 2. Load Metadata
df_clusters = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "clustered_markets.csv"))

# 3. Filter for Valid Markets (Must have Price/Trade data)
# We check the 'prices' folder to ensure we only analyze markets we can actually backtest
con = duckdb.connect()
valid_tokens = con.execute(f"SELECT distinct token_id FROM read_parquet('{RAW_DATA_DIR}/prices/**/*.parquet')").df()
valid_token_set = set(valid_tokens['token_id'])

def has_data(token_str):
    try:
        tokens = json.loads(token_str)
        return any(t in valid_token_set for t in tokens)
    except:
        return False

df_active = df_clusters[df_clusters['clobTokenIds'].apply(has_data)].copy()
print(f"Markets with Data: {len(df_active)} / {len(df_clusters)}")

In [ ]:
def discover_relationships(df):
    relationships = []
    unique_clusters = sorted(df['cluster_id'].unique())

    print(f"Scanning {len(unique_clusters)} clusters for relationships...")

    for cid in unique_clusters:
        # Get markets in this cluster
        subset = df[df['cluster_id'] == cid]

        # Skip small clusters (need at least 2 to make a pair)
        if len(subset) < 2:
            continue

        # Format the input list for the prompt
        # We include ID to map back later, and Question for the AI
        market_list = []
        for _, row in subset.iterrows():
            market_list.append(f"ID {row['conditionId']}: {row['question']}")
        market_list_str = "\n".join(market_list)

        # Prompt (Appendix A)
        prompt = f"""
        Given a list of bets expressed as questions, find pairs of bets whose outcomes are very likely to be related to each other.

        Here is the list:
        {market_list_str}

        For each pair you propose, return a JSON object with:
        - "id_a": The ID of the first market.
        - "id_b": The ID of the second market.
        - "is_same_outcome": Boolean. True if outcomes are likely to be the same. False if likely to be different.
        - "confidence_score": Float between 0.0 and 1.0.
        - "rationale": A string justifying the relationship.

        Return ONLY a JSON list.
        """

        try:
            response = model.generate_content(prompt)
            text = response.text.strip()

            # Extract JSON from markdown code blocks if present
            if "```json" in text:
                text = text.split("```json")[1].split("```")[0]
            elif "```" in text:
                text = text.split("```")[1].split("```")[0]

            pairs = json.loads(text)

            # Validate and Add
            for p in pairs:
                p['cluster_id'] = int(cid)
                relationships.append(p)

            print(f"Cluster {cid}: Found {len(pairs)} pairs.")
            time.sleep(2) # Rate limit safety

        except Exception as e:
            print(f"Cluster {cid}: Error or No Relationships ({e})")

    return pd.DataFrame(relationships)

# Execute
df_relationships = discover_relationships(df_active)

# Save immediately
df_relationships.to_csv(os.path.join(PROCESSED_DATA_DIR, "discovered_relationships.csv"), index=False)
print(f"\nDiscovery Complete. Found {len(df_relationships)} pairs.")
display(df_relationships.head())

In [ ]:
import pandas as pd
import duckdb
import os
import glob
import json

# Initialize DuckDB
con = duckdb.connect()

# 1. Load Target Markets
print("Loading Target Markets...")
df_targets = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "target_markets.csv"))

# 2. Build Trade File Index (Speed Optimization)
print("Indexing Trade Files...")
trade_file_map = {}
try:
    # Scan all trade files to map conditionId -> filename
    con.execute(f"CREATE OR REPLACE VIEW all_trades_scan AS SELECT conditionId, filename FROM read_parquet('{RAW_DATA_DIR}/trades/**/*.parquet', filename=true)")
    mapping_df = con.execute("SELECT DISTINCT conditionId, filename FROM all_trades_scan").df()

    for _, row in mapping_df.iterrows():
        cid = str(row['conditionId']).strip().lower()
        fname = row['filename']
        if cid not in trade_file_map:
            trade_file_map[cid] = []
        trade_file_map[cid].append(fname)
    print(f"Indexed {len(trade_file_map)} markets.")
except Exception as e:
    print(f"Indexing failed: {e}")

# 3. Define Resolution Finder (Corrected Logic)
def get_res_timestamp(condition_id):
    """
    Determines resolution by finding the LAST trade.
    Uses 'outcome' column from trade data to determine winner.
    """
    cid = str(condition_id).strip().lower()

    if cid in trade_file_map:
        files = trade_file_map[cid]
        files_str = ", ".join([f"'{f}'" for f in files])
        source = f"read_parquet([{files_str}])"
    else:
        return None, None, None

    try:
        # Get the very last trade
        query = f"""
        SELECT price, timestamp, outcome
        FROM {source}
        WHERE lower(conditionId) = '{cid}'
        ORDER BY timestamp DESC
        LIMIT 1
        """
        res = con.execute(query).fetchone()

        if res:
            last_price = res[0]
            last_ts = res[1]
            trade_outcome = res[2] # "Yes" or "No"

            winner = "Ambiguous"

            # Logic: Who won?
            if last_price >= 0.9:
                # The traded outcome won
                winner = trade_outcome
            elif last_price <= 0.1:
                # The traded outcome lost, so the OTHER one won
                if trade_outcome == "Yes": winner = "No"
                elif trade_outcome == "No": winner = "Yes"

            return winner, last_ts, last_price

        return None, None, None
    except:
        return None, None, None

# 4. Execute Batch Processing
print("Calculating Resolutions for all markets...")
resolutions = []

for idx, row in df_targets.iterrows():
    cid = row['conditionId']
    outcome, ts, price = get_res_timestamp(cid)

    resolutions.append({
        'conditionId': cid,
        'resolution_outcome': outcome,
        'resolution_timestamp': ts,
        'final_price': price
    })

    if idx % 50 == 0:
        print(f"Processed {idx}/{len(df_targets)}...")

# 5. Save Materialized View
df_resolutions = pd.DataFrame(resolutions)
df_final = pd.merge(df_targets, df_resolutions, on='conditionId', how='left')

output_file = os.path.join(PROCESSED_DATA_DIR, "market_resolutions.csv")
df_final.to_csv(output_file, index=False)
print(f"Resolution Table saved to: {output_file}")
display(df_final[['question', 'resolution_outcome', 'resolution_timestamp', 'final_price']])

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# 1. Load Data
print("Loading Data...")
df_relationships = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "discovered_relationships.csv"))
df_resolutions = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "market_resolutions.csv"))
df_clusters = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "clustered_markets.csv"))

# Create Lookups
res_map = df_resolutions.set_index('conditionId').to_dict('index')
cluster_label_map = df_clusters.set_index('cluster_id')['cluster_label'].to_dict()

# Helper for Entry Price
def get_entry_trade(condition_id, leader_ts):
    cid = str(condition_id).strip().lower()
    if cid in trade_file_map:
        files = trade_file_map[cid]
        files_str = ", ".join([f"'{f}'" for f in files])
        source = f"read_parquet([{files_str}])"
    else:
        return None

    try:
        query = f"""
        SELECT price, timestamp, outcome
        FROM {source}
        WHERE lower(conditionId) = '{cid}'
        AND timestamp > {leader_ts}
        ORDER BY timestamp ASC
        LIMIT 1
        """
        res = con.execute(query).fetchone()
        return res if res else None
    except:
        return None

# 2. Run Backtest
trade_log = []
print(f"Backtesting {len(df_relationships)} pairs...")

for idx, row in df_relationships.iterrows():
    if row['confidence_score'] < 0.5: continue

    id_a = row['id_a']
    id_b = row['id_b']
    relation = row['is_same_outcome']

    res_a = res_map.get(id_a)
    res_b = res_map.get(id_b)

    if not res_a or not res_b: continue
    if res_a['resolution_outcome'] == 'Ambiguous' or res_b['resolution_outcome'] == 'Ambiguous': continue
    if pd.isna(res_a['resolution_timestamp']) or pd.isna(res_b['resolution_timestamp']): continue

    ts_a = int(res_a['resolution_timestamp'])
    ts_b = int(res_b['resolution_timestamp'])

    if ts_a < ts_b:
        leader, follower = res_a, res_b
        leader_id, follower_id = id_a, id_b
        leader_ts = ts_a
    else:
        leader, follower = res_b, res_a
        leader_id, follower_id = id_b, id_a
        leader_ts = ts_b

    target_bet = None
    if relation:
        target_bet = leader['resolution_outcome']
    else:
        target_bet = "No" if leader['resolution_outcome'] == "Yes" else "Yes"

    entry = get_entry_trade(follower_id, leader_ts)
    if not entry: continue

    entry_price_raw, entry_ts, entry_outcome = entry
    entry_price_yes = entry_price_raw if entry_outcome == "Yes" else (1.0 - entry_price_raw)

    if entry_price_yes < 0.10 or entry_price_yes > 0.90: continue

    is_win = (target_bet == follower['resolution_outcome'])
    payout = 1.0 if is_win else 0.0
    cost = entry_price_yes if target_bet == "Yes" else (1.0 - entry_price_yes)
    profit = payout - cost

    trade_log.append({
        'pair_id': idx,
        'leader_id': leader_id,
        'leader_question': leader['question'],
        'follower_id': follower_id,
        'follower_question': follower['question'],
        'gap_seconds': entry_ts - leader_ts,
        'relationship': 'Same' if relation else 'Diff',
        'signal': target_bet,
        'entry_price_yes': entry_price_yes,
        'cost_basis': cost,
        'outcome': follower['resolution_outcome'],
        'profit': profit,
        'cluster_label': cluster_label_map.get(row['cluster_id'], 'unknown'),
        'month': datetime.utcfromtimestamp(entry_ts).strftime('%Y-%m')
    })

# 3. Reporting Engine
df_trades = pd.DataFrame(trade_log)
df_trades.to_csv(os.path.join(PROCESSED_DATA_DIR, "final_trade_log.csv"), index=False)

if not df_trades.empty:
    print(f"\n" + "="*30)
    print(f"GLOBAL PERFORMANCE")
    print(f"="*30)
    total_trades = len(df_trades)
    total_profit = df_trades['profit'].sum()
    total_invested = df_trades['cost_basis'].sum()
    win_rate = (len(df_trades[df_trades['profit'] > 0]) / total_trades) * 100
    roi = (total_profit / total_invested) * 100

    print(f"Trades Executed: {total_trades}")
    print(f"Win Rate:        {win_rate:.2f}%")
    print(f"Total Profit:    ${total_profit:.2f}")
    print(f"Total ROI:       {roi:.2f}%")
    print(f"Avg Gap:         {df_trades['gap_seconds'].mean() / 60:.1f} minutes")

    print(f"\n" + "="*30)
    print(f"CATEGORY BREAKDOWN")
    print(f"="*30)
    # Group by category and calculate metrics
    cat_stats = df_trades.groupby('cluster_label').agg({
        'profit': ['count', 'sum'],
        'cost_basis': 'sum'
    })
    cat_stats.columns = ['Trades', 'Profit', 'Total_Cost']
    cat_stats['ROI %'] = (cat_stats['Profit'] / cat_stats['Total_Cost']) * 100
    print(cat_stats[['Trades', 'Profit', 'ROI %']].sort_values(by='Profit', ascending=False).to_string())

    print(f"\n" + "="*30)
    print(f"MONTHLY PROGRESSION")
    print(f"="*30)
    monthly_stats = df_trades.groupby('month').agg({
        'profit': 'sum',
        'cost_basis': 'sum'
    })
    monthly_stats['ROI %'] = (monthly_stats['profit'] / monthly_stats['cost_basis']) * 100
    print(monthly_stats[['profit', 'ROI %']].to_string())

else:
    print("No trades executed. Check data coverage or confidence thresholds.")